<a href="https://colab.research.google.com/github/Huii0529/MSc_STQD6324_DataManagement_Assignment2/blob/main/P167347_STQD6324_DataManagement_Assignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Mount to Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### **The average rating for each movie & Top ten movies with the highest average ratings.**

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import Row
from pyspark.sql import functions

def loadMovieNames():
    movieNames = {}
    with open("/content/drive/MyDrive/Data Management/ml-100k/u.item", encoding='ISO-8859-1') as f:
        for line in f:
            fields = line.split('|')
            movieNames[int(fields[0])] = fields [1]
    return movieNames

def parseInput(line):
    fields = line.split()
    return Row(movieID = int(fields[1]), rating = float(fields[2]))

if __name__ == "__main__":
    # Create a SparkSession
    spark = SparkSession.builder.appName("PopularMovies").getOrCreate()

    # Load up our movie ID -> name dictionary
    movieNames = loadMovieNames()

    # Get the raw data
    lines = spark.sparkContext.textFile("/content/drive/MyDrive/Data Management/ml-100k/u.data")

    # Convert it to a RDD of Row objects with (movieID, rating)
    movies = lines.map(parseInput)

    # Convert that to a DataFrame
    movieDataset = spark.createDataFrame(movies)

    # Compute average rating for each movieID
    averageRatings = movieDataset.groupBy("movieID").avg("rating")

    # Compute count of ratings for each movieID
    counts = movieDataset.groupBy("movieID").count()

    # Join the two together
    averagesAndCounts = counts.join(averageRatings, "movieID")

    # Show the top 10 best movies results
    bestTen = averagesAndCounts.orderBy(functions.desc("avg(rating)")).take(10)

    # Print them out, convert movie ID's to names as we go
    print("The Top Ten Highest Ranking Movies")
    for movie in bestTen:
        print(movieNames[movie[0]], movie[1], movie[2])

    # Stop the session
    spark.stop()

The Top Ten Highest Ranking Movies
Aiqing wansui (1994) 1 5.0
Prefontaine (1997) 3 5.0
Someone Else's America (1995) 1 5.0
Star Kid (1997) 3 5.0
Great Day in Harlem, A (1994) 1 5.0
Entertaining Angels: The Dorothy Day Story (1996) 1 5.0
Santa with Muscles (1996) 2 5.0
They Made Me a Criminal (1939) 1 5.0
Marlene Dietrich: Shadow and Light (1996)  1 5.0
Saint of Fort Washington, The (1993) 2 5.0


### **Users who have rated at least 50 movies and their favourite movie genre**

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import Row
from pyspark.sql import functions as F

# Load movie names
def loadMovieNames():
    movieNames = {}

    with open("/content/drive/MyDrive/Data Management/ml-100k/u.item", encoding='ISO-8859-1') as f:
        for line in f:
            fields = line.strip().split('|')
            movieNames[int(fields[0])] = fields[1]

    return movieNames

# Load movie genres into a dictionary
def loadMovieGenres():
    genreNames = [
        "Unknown", "Action", "Adventure", "Animation",
        "Children's", "Comedy", "Crime", "Documentary",
        "Drama", "Fantasy", "Film-Noir", "Horror",
        "Musical", "Mystery", "Romance", "Sci-Fi",
        "Thriller", "War", "Western"
    ]

    movieGenres = {}

    with open("/content/drive/MyDrive/Data Management/ml-100k/u.item", encoding='ISO-8859-1') as f:
        for line in f:
            fields = line.strip().split('|')

            genres=[]

            for i in range(19):
                if int(fields[5 + i]) == 1:
                    genres.append(genreNames[i])

            movieGenres[int(fields[0])] = genres

        return movieGenres

# Parse ratings
def parseRatings(line):
    fields = line.split()

    return Row(
        userID=int(fields[0]),
        movieID=int(fields[1]),
        rating=float(fields[2])
    )

if __name__ == "__main__":

    spark = SparkSession.builder.appName("FavouriteGenre").getOrCreate()

    # Read ratings from HDFS
    lines = spark.sparkContext.textFile(
        "/content/drive/MyDrive/Data Management/ml-100k/u.data")

    ratings = lines.map(parseRatings)
    movieDataset = spark.createDataFrame(ratings)

    # Load movie information
    movieNames = loadMovieNames()
    movieGenres = loadMovieGenres()

    # Users who rated at least 50 movies
    activeUsers = (movieDataset.groupby("userID").count()
                   .filter(F.col("count") >= 50)
                   .orderBy(F.desc("count"))
                   .limit(10)
    )

    print("\nTop 10 Most Active Users and Their Favourite Genres\n")

    for user in activeUsers.collect():

        userID = user["userID"]
        totalMovies = user["count"]

        # Get all ratings for this user
        userRatings = (movieDataset.filter(F.col("userID") == userID).collect())

        genreCount = {}

        for rating in userRatings:
            movieID = rating.movieID

            if movieID in movieGenres:
                for genre in movieGenres[movieID]:
                    genreCount[genre] = genreCount.get(genre, 0) + 1


        if len(genreCount) > 0:
            favouriteGenre = max(genreCount, key=genreCount.get)

            print(
                "User:", userID,
                "| Movies Rated:", totalMovies,
                "| Favourite Genre:", favouriteGenre,
                "| Total Genre Movies Rated:", genreCount[favouriteGenre]
            )

# Stop the session
spark.stop()


Top 10 Most Active Users and Their Favourite Genres

User: 405 | Movies Rated: 737 | Favourite Genre: Drama | Total Genre Movies Rated: 309
User: 655 | Movies Rated: 685 | Favourite Genre: Drama | Total Genre Movies Rated: 410
User: 13 | Movies Rated: 636 | Favourite Genre: Drama | Total Genre Movies Rated: 218
User: 450 | Movies Rated: 540 | Favourite Genre: Drama | Total Genre Movies Rated: 237
User: 276 | Movies Rated: 518 | Favourite Genre: Drama | Total Genre Movies Rated: 168
User: 416 | Movies Rated: 493 | Favourite Genre: Drama | Total Genre Movies Rated: 212
User: 537 | Movies Rated: 490 | Favourite Genre: Drama | Total Genre Movies Rated: 251
User: 303 | Movies Rated: 484 | Favourite Genre: Comedy | Total Genre Movies Rated: 184
User: 234 | Movies Rated: 480 | Favourite Genre: Drama | Total Genre Movies Rated: 213
User: 393 | Movies Rated: 448 | Favourite Genre: Comedy | Total Genre Movies Rated: 191


### **Install Java**

In [ ]:
!apt-get update
!apt-get install -y openjdk-11-jdk

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:6 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  at-spi2-core fonts-dejavu-core fonts-de

In [ ]:
!update-java-alternatives -l

java-1.11.0-openjdk-amd64      1111       /usr/lib/jvm/java-1.11.0-openjdk-amd64
java-1.17.0-openjdk-amd64      1711       /usr/lib/jvm/java-1.17.0-openjdk-amd64


In [ ]:
!sudo update-java-alternatives -s java-1.11.0-openjdk-amd64

In [ ]:
!java -version

openjdk version "11.0.31" 2026-04-21
OpenJDK Runtime Environment (build 11.0.31+11-post-1ubuntu1-22.04.2-Ubuntu)
OpenJDK 64-Bit Server VM (build 11.0.31+11-post-1ubuntu1-22.04.2-Ubuntu, mixed mode, sharing)


In [ ]:
!wget https://archive.apache.org/dist/cassandra/4.1.11/apache-cassandra-4.1.11-bin.tar.gz
!tar -xzf apache-cassandra-4.1.11-bin.tar.gz

--2026-07-08 00:53:29--  https://archive.apache.org/dist/cassandra/4.1.11/apache-cassandra-4.1.11-bin.tar.gz
Resolving archive.apache.org (archive.apache.org)... 65.108.204.189, 2a01:4f9:1a:a084::2
Connecting to archive.apache.org (archive.apache.org)|65.108.204.189|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 53640694 (51M) [application/x-gzip]
Saving to: ‘apache-cassandra-4.1.11-bin.tar.gz’

apache-cassandra-4. 100%[===================>]  51.16M  13.9MB/s    in 6.5s    

2026-07-08 00:53:35 (7.83 MB/s) - ‘apache-cassandra-4.1.11-bin.tar.gz’ saved [53640694/53640694]



In [ ]:
!apache-cassandra-4.1.11/bin/cassandra -R

OpenJDK 64-Bit Server VM warning: Option UseConcMarkSweepGC was deprecated in version 9.0 and will likely be removed in a future release.
CompileCommand: dontinline org/apache/cassandra/db/Columns$Serializer.deserializeLargeSubset(Lorg/apache/cassandra/io/util/DataInputPlus;Lorg/apache/cassandra/db/Columns;I)Lorg/apache/cassandra/db/Columns;
CompileCommand: dontinline org/apache/cassandra/db/Columns$Serializer.serializeLargeSubset(Ljava/util/Collection;ILorg/apache/cassandra/db/Columns;ILorg/apache/cassandra/io/util/DataOutputPlus;)V
CompileCommand: dontinline org/apache/cassandra/db/Columns$Serializer.serializeLargeSubsetSize(Ljava/util/Collection;ILorg/apache/cassandra/db/Columns;I)I
CompileCommand: dontinline org/apache/cassandra/db/commitlog/AbstractCommitLogSegmentManager.advanceAllocatingFrom(Lorg/apache/cassandra/db/commitlog/CommitLogSegment;)V
CompileCommand: dontinline org/apache/cassandra/db/transform/BaseIterator.tryGetMoreContents()Z
CompileCommand: dontinline org/apache/c

In [ ]:
!apache-cassandra-4.1.11/bin/nodetool status

Datacenter: datacenter1
Status=Up/Down
|/ State=Normal/Leaving/Joining/Moving
--  Address    Load       Tokens  Owns (effective)  Host ID                               Rack 
UN  127.0.0.1  104.4 KiB  16      100.0%            e59b4922-e434-4b57-8504-e519ad72c348  rack1



In [ ]:
!ls

apache-cassandra-4.1.11		    drive
apache-cassandra-4.1.11-bin.tar.gz  ml-100k.zip
apache-cassandra-4.1.8		    sample_data
apache-cassandra-4.1.8-bin.tar.gz


In [ ]:
!pip install cassandra-driver

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 71.3 MB/s eta 0:00:00


In [ ]:
!apache-cassandra-4.1.8/bin/cqlsh

/bin/bash: line 1: apache-cassandra-4.1.8/bin/cqlsh: No such file or directory


In [ ]:
cqlsh

# Create database
CREATE KEYSPACE movielens WITH replication = {'class': 'SimpleStrategy'},
'replication_factor':'1'} AND durable_writes = true;

USE movielens;
CREATE TABLE users (userID int, age int, gender text, occupation text, zip text, PRIMARY KEY (user_id));
DESCRIBE TABLE users;
SELECT * FROM users;
exit

SyntaxError: unmatched '}' (3188105304.py, line 5)

### **Users who are less than 20 years old**

In [ ]:
from os import read
from pyspark.sql import SparkSession
from pyspark.sql import Row
from pyspark.sql import functions

def parseInput(line):
  fields = line.split('|')
  return Row(userID = int(fields[0]), age = int(fields[1]),
             gender = fields[2], occupation = fields[3],
             zip = fields[4])

if __name__ == "__main__":
  cassandra_connector_package = "com.datastax.spark:spark-cassandra-connector_2.12:3.4.0" # Adjust version as needed

  # Create a SparkSession with Cassandra Connector
  spark = SparkSession.builder \
    .appName("CassandraIntegration") \
    .config("spark.cassandra.connection.host", "127.0.0.1") \
    .config("spark.jars.packages", cassandra_connector_package) \
    .getOrCreate()

  # Get the raw data
  lines = spark.sparkContext.textFile("/content/drive/MyDrive/Data Management/ml-100k/u.user")

  # Convert it to a RDD of Row objects with the schema
  users = lines.map(parseInput)

  # Convert that to a DataFrame
  usersDataset = spark.createDataFrame(users)

  # Write it to Cassandra
  usersDataset.write\
    .format("org.apache.spark.sql.cassandra")\
    .mode('append')\
    .options(table="users", keyspace="movielens")\
    .save()

  # Read it back from Cassandra into a new Dataframe
  readUsers = spark.read\
    .format("org.apache.spark.sql.cassandra")\
    .options(table="users", keyspace="movielens")\
    .load()

  readUsers.createOrReplaceTempView("users")

  sqlDF = spark.sql("SELECT * FROM users WHERE age < 20")
  sqlDF.show()

  # Stop the session
  spark.stop()

Py4JJavaError: An error occurred while calling o425.save.
: org.apache.spark.SparkClassNotFoundException: [DATA_SOURCE_NOT_FOUND] Failed to find the data source: org.apache.spark.sql.cassandra. Make sure the provider name is correct and the package is properly registered and compatible with your Spark version. SQLSTATE: 42K02
	at org.apache.spark.sql.errors.QueryExecutionErrors$.dataSourceNotFoundError(QueryExecutionErrors.scala:722)
	at org.apache.spark.sql.execution.datasources.DataSource$.lookupDataSource(DataSource.scala:681)
	at org.apache.spark.sql.execution.datasources.DataSource$.lookupDataSourceV2(DataSource.scala:740)
	at org.apache.spark.sql.classic.DataFrameWriter.lookupV2Provider(DataFrameWriter.scala:626)
	at org.apache.spark.sql.classic.DataFrameWriter.saveInternal(DataFrameWriter.scala:135)
	at org.apache.spark.sql.classic.DataFrameWriter.save(DataFrameWriter.scala:126)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
	at java.base/jdk.

### **Users who are Scientist and age is between 30 and 40 years old.**

In [ ]:
from os import read
from pyspark.sql import SparkSession
from pyspark.sql import Row
from pyspark.sql import functions

def parseInput(line):
  fields = line.split('|')
  return Row(userID = int(fields[0]), age = int(fields[1]),
             gender = fields[2], occupation = fields[3],
             zip = fields[4])

if __name__ == "__main__":
  cassandra_connector_package = "com.datastax.spark:spark-cassandra-connector_2.12:3.4.0" # Adjust version as needed

  # Create a SparkSession with Cassandra Connector
  spark = SparkSession.builder \
    .appName("CassandraIntegration") \
    .config("spark.cassandra.connection.host", "127.0.0.1") \
    .config("spark.jars.packages", cassandra_connector_package) \
    .getOrCreate()

  # Get the raw data
  lines = spark.sparkContext.textFile("/content/drive/MyDrive/Data Management/ml-100k/u.user")

  # Convert it to a RDD of Row objects with the schema
  users = lines.map(parseInput)

  # Convert that to a DataFrame
  usersDataset = spark.createDataFrame(users)

  # Write it to Cassandra
  usersDataset.write\
    .format("org.apache.spark.sql.cassandra")\
    .mode('append')\
    .options(table="users", keyspace="movielens")\
    .save()

  # Read it back from Cassandra into a new Dataframe
  readUsers = spark.read\
    .format("org.apache.spark.sql.cassandra")\
    .options(table="users", keyspace="movielens")\
    .load()

  readUsers.createOrReplaceTempView("users")

  sqlDF = spark.sql("SELECT * FROM users WHERE occupation = 'Scientist' AND age >= 30 AND age <= 40")
  sqlDF.show()

  # Stop the session
  spark.stop()